Дорогой студент!

В домашнем задании Lite вам предлагается поработать подробнее с параметрами словаря и формированием гиперпараметров нейронной сети. Создайте 9 нейросетей с различными гиперпараметрами (см. пунтк 2 и 3)

 Для этого необходимо:

  1. Воссоздать ноутбук, аналогичный ноутбуку практической части №1, загрузив при этом необходимую нам базу (код уже доступен в ноутбуке).

  2. Задать в ноутбуке следующие параметры для размера словаря, ширины окна и шага:

    - Размер словаря - от 10000 до 20000 (выбрать меньшее значение диапазона, если будет перегрузка ОЗУ и перезапуск подключения к Colaboratory)
    - Ширина окна - от 1000 до 2000
    - Шаг - от 100 до 500 (на обучение лучше влияет наименьший шаг, но это может перегрузить ОЗУ).

  3. Создать архитектуру сети и задать гиперпараметры. Можно воспользоваться шаблоном:
  
   - Добавьте модель прямого распространения **Sequential()**
   - Добавьте один или несколько полносвязных (**Dense**) слоёв
   - Добавьте слои **Dropout()** и **BatchNormalization()**
   - Добавьте выходной полносвязный слой с количеством нейронов, соответствующим количеству классов (число писателей)
  
   Напомним, что точность сети можно проверить по значению показателя 'val_accuracy' на конце каждой эпохи.
   

In [ ]:
import numpy as np
from tensorflow.keras import utils
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.text import Tokenizer

from sklearn.metrics import confusion_matrix
import gdown, os, re

In [ ]:
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l7/writers.zip', None, quiet=True)
!unzip -qo writers.zip -d writers/

FILE_DIR  = 'writers'
SIG_TRAIN = 'обучающая'
SIG_TEST  = 'тестовая'

читаем файлы из папки, извлекаем класс и тип (train/test) из имени,
группируем тексты по классам и объединяем их в два списка: text_train и text_test

In [ ]:
CLASS_LIST = []
text_train = []
text_test = []

for file_name in os.listdir(FILE_DIR):
    m = re.match(r'\((.+)\) (\S+)_', file_name)
    if m:
        class_name = m[1]
        subset_name = m[2].lower()

        is_train = SIG_TRAIN in subset_name
        is_test = SIG_TEST in subset_name

        if is_train or is_test:
            if class_name not in CLASS_LIST:
                CLASS_LIST.append(class_name)
                text_train.append('')
                text_test.append('')

            cls = CLASS_LIST.index(class_name)

            with open(f'{FILE_DIR}/{file_name}', 'r') as f:
                text = f.read()

            if is_train:
                text_train[cls] += ' ' + text.replace('\n', ' ')
            else:
                text_test[cls] += ' ' + text.replace('\n', ' ')

CLASS_COUNT = len(CLASS_LIST)

In [ ]:
VOCAB_SIZES = [5000, 10000, 15000]
WIN_SIZES   = [1000, 1500, 2000]
WIN_HOP = 100

In [ ]:
class TextClassifier:
    def __init__(self, text_train, text_test, class_count):
        self.text_train = text_train
        self.text_test = text_test
        self.class_count = class_count

    #1. токенизация
    def tokenize(self, vocab_size):
        self.tokenizer = Tokenizer(num_words=vocab_size, oov_token='UNK') # словарь + токен для неизвестных слов
        self.tokenizer.fit_on_texts(self.text_train) # строим словарь по train

        self.seq_train = self.tokenizer.texts_to_sequences(self.text_train) # текст → последовательности индексов
        self.seq_test  = self.tokenizer.texts_to_sequences(self.text_test)

    #2. разбиение
    def split_sequence(self, sequence, win_size, hop):
        return [
            sequence[i:i + win_size]  # окно фиксированной длины
            for i in range(0, len(sequence) - win_size + 1, hop) # сдвиг
        ]

    #3. формирование выборок
    def vectorize(self, seq_list, win_size, hop):
        x, y = [], []
        for cls in range(len(seq_list)):
            vectors = self.split_sequence(seq_list[cls], win_size, hop) # окна
            x += vectors
            y += [utils.to_categorical(cls, self.class_count)] * len(vectors) # one-hot метки
        return np.array(x), np.array(y)

    #4. подготовка данных
    def prepare_data(self, win_size, hop):
        self.x_train, self.y_train = self.vectorize(self.seq_train, win_size, hop) #режем на окна, формируем признаки (x) и one-hot метки (y)
        self.x_test,  self.y_test  = self.vectorize(self.seq_test, win_size, hop)

        # BoW
        self.x_train = self.tokenizer.sequences_to_matrix(self.x_train.tolist())
        self.x_test  = self.tokenizer.sequences_to_matrix(self.x_test.tolist())

    #5. модель
    def build_model(self, vocab_size):
        model = Sequential([
            Dense(256, activation='relu', input_dim=vocab_size),
            Dropout(0.3),
            BatchNormalization(),

            Dense(128, activation='relu'),
            Dropout(0.3),

            Dense(self.class_count, activation='softmax')
        ])

        model.compile(
            optimizer='adam',
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )

        return model

    # --- 6. обучение ---
    def train(self, vocab_size, win_size, hop):
        self.tokenize(vocab_size)
        self.prepare_data(win_size, hop)

        model = self.build_model(vocab_size)

        history = model.fit(
            self.x_train, self.y_train,
            epochs=5,
            batch_size=128,
            validation_data=(self.x_test, self.y_test),
            verbose=0
        )

        val_acc = history.history['val_accuracy'][-1]
        return val_acc

In [ ]:
classifier = TextClassifier(text_train, text_test, CLASS_COUNT)

results = []

for vocab in [10000, 15000, 20000]:
    for win in [1000, 1500, 2000]:
        acc = classifier.train(vocab, win, 100)

        results.append({
            "vocab": vocab,
            "win": win,
            "hop": 100,
            "val_acc": acc
        })

        print(f"VOCAB={vocab}, WIN={win} → {acc:.4f}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


VOCAB=10000, WIN=1000 → 0.9040
VOCAB=10000, WIN=1500 → 0.9298
VOCAB=10000, WIN=2000 → 0.9185
VOCAB=15000, WIN=1000 → 0.9116
VOCAB=15000, WIN=1500 → 0.9246
VOCAB=15000, WIN=2000 → 0.9239
VOCAB=20000, WIN=1000 → 0.9152
VOCAB=20000, WIN=1500 → 0.9285
VOCAB=20000, WIN=2000 → 0.9395


In [ ]:
sorted(results, key=lambda x: x["val_acc"], reverse=True)

[{'vocab': 20000, 'win': 2000, 'hop': 100, 'val_acc': 0.9394904375076294},
 {'vocab': 10000, 'win': 1500, 'hop': 100, 'val_acc': 0.9298068881034851},
 {'vocab': 20000, 'win': 1500, 'hop': 100, 'val_acc': 0.9285096526145935},
 {'vocab': 15000, 'win': 1500, 'hop': 100, 'val_acc': 0.9246180653572083},
 {'vocab': 15000, 'win': 2000, 'hop': 100, 'val_acc': 0.9238563776016235},
 {'vocab': 10000, 'win': 2000, 'hop': 100, 'val_acc': 0.9185003042221069},
 {'vocab': 20000, 'win': 1000, 'hop': 100, 'val_acc': 0.9151837229728699},
 {'vocab': 15000, 'win': 1000, 'hop': 100, 'val_acc': 0.91159588098526},
 {'vocab': 10000, 'win': 1000, 'hop': 100, 'val_acc': 0.9039896726608276}]